# 🔍 Sistem Cerdas Deteksi Penipuan Lowongan Kerja Palsu
**Pipeline end-to-end: Data → EDA → Preprocessing → 3 Model Alternatif → Evaluasi → Pemilihan Model Terbaik → Export (Deploy)**

Dataset: `fake_job_postings.csv` (Employment Scam Aegean Dataset, ±17.880 lowongan, label `fraudulent` 0 = asli, 1 = palsu).

**Tiga model alternatif** (semuanya berupa *scikit-learn Pipeline* lengkap dari data mentah sampai prediksi):

| Model | Keluarga | Alasan dipilih |
|---|---|---|
| A. TF-IDF + **Logistic Regression** | Linear | Baseline kuat untuk teks, cepat, mudah diinterpretasi |
| B. TF-IDF + **Linear SVM (terkalibrasi)** | Margin maksimum | Biasanya sangat baik pada teks berdimensi tinggi |
| C. TF-IDF + **Random Forest** | Ensemble pohon | Menangkap interaksi non-linier antar fitur teks & metadata |

**Catatan metodologi:** data sangat tidak seimbang (~5% palsu), jadi *accuracy* menyesatkan. Evaluasi utama memakai **Precision, Recall, F1 (kelas palsu), PR-AUC**, dan ambang keputusan (*threshold*) dituning lewat cross-validation pada data latih (bukan pada data uji) agar tidak terjadi kebocoran data.

## 1. Setup & Import Library

In [ ]:
import warnings, re, json, shutil, os, glob, datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("scikit-learn:", sklearn.__version__)

## 2. Memuat Dataset
Jalankan sel di bawah, lalu **upload file `fake_job_postings.csv.zip`** (atau `.csv`) saat diminta.  
Alternatif: taruh file di Google Drive lalu ubah `DATA_PATH` secara manual.

In [ ]:
DATA_PATH = None   # contoh manual: "/content/drive/MyDrive/fake_job_postings.csv.zip"

if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()               # pilih fake_job_postings.csv.zip
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:                         # dijalankan di luar Colab
        DATA_PATH = (glob.glob("fake_job_postings*") or glob.glob("**/fake_job_postings*", recursive=True))[0]

print("Membaca:", DATA_PATH)
df = pd.read_csv(DATA_PATH)                     # pandas otomatis membuka .zip
print("Ukuran data:", df.shape)
df.head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Distribusi label (0 = asli, 1 = palsu):")
print(df["fraudulent"].value_counts())
print(f"\nProporsi palsu: {df['fraudulent'].mean():.2%}")
print(f"Duplikat (selain job_id): {df.duplicated(subset=[c for c in df.columns if c != 'job_id']).sum()}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

sns.countplot(x="fraudulent", data=df, ax=ax[0], palette=["#4C78A8", "#E45756"])
ax[0].set_title("Distribusi Kelas (sangat tidak seimbang)")
ax[0].set_xticklabels(["Asli (0)", "Palsu (1)"])

miss = df.drop(columns="fraudulent").isna().mean().sort_values(ascending=False)
miss[miss > 0].plot.barh(ax=ax[1], color="#F58518")
ax[1].set_title("Proporsi Missing Value per Kolom"); ax[1].invert_yaxis()

rate = df.groupby(["has_company_logo", "has_questions"])["fraudulent"].mean().unstack()
sns.heatmap(rate, annot=True, fmt=".1%", cmap="Reds", ax=ax[2])
ax[2].set_title("Tingkat Palsu: logo perusahaan × pertanyaan screening")
plt.tight_layout(); plt.show()

In [ ]:
# Perbandingan panjang teks & kelengkapan informasi antara lowongan asli vs palsu
tmp = df.copy()
tmp["len_desc"] = tmp["description"].fillna("").str.len()
tmp["tanpa_profil_perusahaan"] = tmp["company_profile"].isna()
tmp["tanpa_gaji"] = tmp["salary_range"].isna()
print(tmp.groupby("fraudulent")[["len_desc", "tanpa_profil_perusahaan", "tanpa_gaji"]].mean().round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
top_c = df["employment_type"].fillna("(kosong)")
(df.assign(employment_type=top_c).groupby("employment_type")["fraudulent"].mean()
   .sort_values().plot.barh(ax=ax[0], color="#E45756"))
ax[0].set_title("Tingkat Palsu per Tipe Pekerjaan")
sns.kdeplot(data=tmp, x=np.log1p(tmp["len_desc"]), hue="fraudulent", common_norm=False, ax=ax[1], fill=True)
ax[1].set_title("Distribusi log(panjang deskripsi)")
plt.tight_layout(); plt.show()

## 4. Modul Preprocessing & Feature Engineering
Seluruh logika pembentukan fitur ditulis ke file **`fake_job_utils.py`** supaya bisa dipakai ulang saat deploy
(model yang di-export akan memanggil fungsi yang sama → hasil training dan inferensi konsisten).

Fitur yang dibentuk:
- **Teks gabungan**: judul + profil perusahaan + deskripsi + persyaratan + benefit (dibersihkan: HTML, entitas, placeholder `#URL_...#`, `#EMAIL_...#`, `#PHONE_...#`).
- **Kategorikal**: negara (dari `location`), tipe pekerjaan, pengalaman, pendidikan, fungsi, industri.
- **Numerik/biner**: panjang tiap bagian teks (log), jumlah tanda seru, rasio huruf kapital, jumlah URL/email/telepon, ada/tidaknya gaji, ada/tidaknya profil, logo, pertanyaan screening, telecommuting, dll.

In [ ]:
%%writefile fake_job_utils.py
import html
import re
import numpy as np
import pandas as pd

TEXT_PARTS = ["title", "company_profile", "description", "requirements", "benefits"]
CAT_COLS = ["country", "employment_type", "required_experience",
            "required_education", "function", "industry"]
NUM_COLS = ["len_title", "len_profile", "len_desc", "len_req", "len_benefits",
            "n_exclaim", "caps_ratio", "n_url", "n_email", "n_phone", "n_dollar",
            "has_salary", "has_department", "no_profile", "no_req", "no_benefits",
            "telecommuting", "has_company_logo", "has_questions"]
TEXT_COL = "text"

# Kolom mentah yang diterima model saat inferensi (kolom yang tidak diisi dianggap kosong)
RAW_COLUMNS = ["title", "location", "department", "salary_range", "company_profile",
               "description", "requirements", "benefits", "telecommuting",
               "has_company_logo", "has_questions", "employment_type",
               "required_experience", "required_education", "industry", "function"]


def _s(x):
    return x if isinstance(x, str) else ""


def _clean_text(s):
    s = html.unescape(_s(s))
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"#URL_[A-Za-z0-9]+#", " urltoken ", s)
    s = re.sub(r"#EMAIL_[A-Za-z0-9]+#", " emailtoken ", s)
    s = re.sub(r"#PHONE_[A-Za-z0-9]+#", " phonetoken ", s)
    s = re.sub(r"(?<=[a-z0-9\.\,\)\!])(?=[A-Z][a-z])", " ", s)   # pisahkan kalimat yang menempel
    s = re.sub(r"[^A-Za-z0-9\s\$\!]", " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()


def prepare_features(df):
    # Mengubah data lowongan mentah (DataFrame) menjadi tabel fitur siap model.
    d = df.copy()
    for c in RAW_COLUMNS:
        if c not in d.columns:
            d[c] = np.nan

    raw_all = d[TEXT_PARTS].apply(lambda col: col.map(_s)).agg(" ".join, axis=1)
    out = pd.DataFrame(index=d.index)
    out[TEXT_COL] = d[TEXT_PARTS].apply(lambda col: col.map(_clean_text)).agg(" ".join, axis=1)

    country = d["location"].map(lambda x: _s(x).split(",")[0].strip().upper() or "UNKNOWN")
    out["country"] = country
    for c in CAT_COLS[1:]:
        out[c] = d[c].fillna("unknown").astype(str)

    out["len_title"] = np.log1p(d["title"].map(lambda x: len(_s(x))))
    out["len_profile"] = np.log1p(d["company_profile"].map(lambda x: len(_s(x))))
    out["len_desc"] = np.log1p(d["description"].map(lambda x: len(_s(x))))
    out["len_req"] = np.log1p(d["requirements"].map(lambda x: len(_s(x))))
    out["len_benefits"] = np.log1p(d["benefits"].map(lambda x: len(_s(x))))
    out["n_exclaim"] = raw_all.str.count("!")
    out["caps_ratio"] = raw_all.map(lambda t: sum(ch.isupper() for ch in t) / max(len(t), 1))
    out["n_url"] = raw_all.str.count(r"#URL_")
    out["n_email"] = raw_all.str.count(r"#EMAIL_")
    out["n_phone"] = raw_all.str.count(r"#PHONE_")
    out["n_dollar"] = raw_all.str.count(r"\$")
    out["has_salary"] = d["salary_range"].notna().astype(int)
    out["has_department"] = d["department"].notna().astype(int)
    out["no_profile"] = (d["company_profile"].map(lambda x: len(_s(x))) == 0).astype(int)
    out["no_req"] = (d["requirements"].map(lambda x: len(_s(x))) == 0).astype(int)
    out["no_benefits"] = (d["benefits"].map(lambda x: len(_s(x))) == 0).astype(int)
    for c in ["telecommuting", "has_company_logo", "has_questions"]:
        out[c] = pd.to_numeric(d[c], errors="coerce").fillna(0).astype(int)
    return out


def predict_postings(bundle, df):
    # Memprediksi sekumpulan lowongan. bundle = hasil joblib.load("fake_job_detector.joblib")
    proba = bundle["model"].predict_proba(df)[:, 1]
    thr = bundle["threshold"]
    res = pd.DataFrame({"probabilitas_palsu": proba.round(4)}, index=df.index)
    res["prediksi"] = np.where(proba >= thr, "PALSU", "ASLI")
    res["tingkat_risiko"] = np.select(
        [proba >= thr, proba >= thr / 2], ["TINGGI", "WASPADA"], default="RENDAH")
    return res

## 5. Preprocessing Data & Pembagian Train/Test

In [ ]:
import importlib, fake_job_utils
importlib.reload(fake_job_utils)
from fake_job_utils import (prepare_features, predict_postings, TEXT_COL, CAT_COLS,
                            NUM_COLS, RAW_COLUMNS)

# 1) Buang duplikat (mencegah kebocoran data: sampel identik muncul di train & test)
dedup_cols = [c for c in df.columns if c != "job_id"]
data = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
print(f"Setelah buang duplikat: {len(data)} baris (dibuang {len(df) - len(data)})")

X = data[RAW_COLUMNS]
y = data["fraudulent"].astype(int)

# 2) Split stratified 80:20 -> proporsi kelas palsu terjaga di train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f"Train: {len(X_train)} (palsu {y_train.mean():.2%}) | Test: {len(X_test)} (palsu {y_test.mean():.2%})")

# Contoh hasil feature engineering
prepare_features(X_train.head(2)).T.head(12)

## 6. Mendefinisikan 3 Model Alternatif
Ketiganya memakai **struktur Pipeline yang sama**: `prepare_features` → (TF-IDF teks + One-Hot kategorikal + scaling numerik) → classifier.
Penanganan ketidakseimbangan kelas memakai `class_weight` (tanpa menggandakan data, sehingga tidak ada kebocoran saat cross-validation).

In [ ]:
def make_preprocessor(max_features):
    return ColumnTransformer([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=max_features, min_df=3,
                                  max_df=0.9, sublinear_tf=True, stop_words="english"), TEXT_COL),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), CAT_COLS),
        ("num", StandardScaler(), NUM_COLS),
    ])

def build_pipeline(classifier, max_features=30000):
    return Pipeline([
        ("features", FunctionTransformer(prepare_features, validate=False)),
        ("prep", make_preprocessor(max_features)),
        ("clf", classifier),
    ])

models = {
    "A. Logistic Regression": build_pipeline(
        LogisticRegression(C=10, class_weight="balanced", solver="liblinear", max_iter=2000,
                           random_state=SEED)),
    "B. Linear SVM (kalibrasi)": build_pipeline(
        CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced", random_state=SEED),
                               method="sigmoid", cv=3)),
    "C. Random Forest": build_pipeline(
        RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                               min_samples_leaf=1, n_jobs=-1, random_state=SEED),
        max_features=5000),
}
print("Model yang akan dibandingkan:", list(models))

## 7. Cross-Validation (5-fold) & Penentuan Threshold
Untuk tiap model dihasilkan **probabilitas out-of-fold (OOF)** pada data latih. Dari OOF ini dihitung metrik perbandingan
dan dicari **threshold optimal (F1 maksimum)**. Data uji belum disentuh sama sekali.

In [ ]:
import time
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def best_threshold(y_true, proba):
    p, r, t = precision_recall_curve(y_true, proba)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-9, None)
    i = int(np.nanargmax(f1))
    return float(t[i]), float(f1[i])

oof_proba, cv_rows, thresholds = {}, [], {}
for name, pipe in models.items():
    t0 = time.time()
    proba = cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict_proba", n_jobs=1)[:, 1]
    thr, f1_best = best_threshold(y_train, proba)
    pred = (proba >= thr).astype(int)
    oof_proba[name], thresholds[name] = proba, thr
    cv_rows.append({
        "Model": name,
        "PR-AUC": average_precision_score(y_train, proba),
        "ROC-AUC": roc_auc_score(y_train, proba),
        "Precision": precision_score(y_train, pred),
        "Recall": recall_score(y_train, pred),
        "F1 (palsu)": f1_score(y_train, pred),
        "Threshold": thr,
        "Waktu (dtk)": time.time() - t0,
    })
    print(f"✔ {name} selesai ({time.time() - t0:.0f} dtk)")

cv_results = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_results

## 8. Evaluasi pada Data Uji (Hold-out)
Setiap model dilatih ulang pada seluruh data latih lalu diuji sekali pada data uji dengan threshold dari langkah 7.

In [ ]:
fitted, test_proba, test_rows = {}, {}, []
for name, pipe in models.items():
    model = clone(pipe).fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    thr = thresholds[name]
    pred = (proba >= thr).astype(int)
    fitted[name], test_proba[name] = model, proba
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 (palsu)": f1_score(y_test, pred),
        "F1 @thr 0.5": f1_score(y_test, (proba >= 0.5).astype(int)),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Threshold": thr,
    })

test_results = pd.DataFrame(test_rows).set_index("Model").round(4)
test_results

In [ ]:
# Visualisasi: confusion matrix, kurva ROC & Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (name, proba) in zip(axes, test_proba.items()):
    pred = (proba >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Asli", "Palsu"]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\n(thr = {thresholds[name]:.2f})", fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for name, proba in test_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    p, r, _ = precision_recall_curve(y_test, proba)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
    ax[1].plot(r, p, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_title("Kurva ROC")
ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate"); ax[0].legend()
ax[1].set_title("Kurva Precision–Recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend()
plt.tight_layout(); plt.show()

## 9. Memilih Model Terbaik
Kriteria: **PR-AUC hasil cross-validation** (metrik paling tepat untuk kelas minoritas, dan dipilih dari data latih agar data uji tetap jujur sebagai konfirmasi akhir).
Silakan juga baca tabel di atas secara manual — misalnya jika Anda lebih mengutamakan *recall* (jangan sampai ada lowongan palsu lolos) atau *precision* (jangan sampai lowongan asli diblokir).

In [ ]:
best_name = cv_results["PR-AUC"].idxmax()
print("🏆 Model terbaik (PR-AUC CV tertinggi):", best_name)

summary = pd.concat({"CV (train)": cv_results[["PR-AUC", "Precision", "Recall", "F1 (palsu)"]],
                     "Test (hold-out)": test_results[["PR-AUC", "Precision", "Recall", "F1 (palsu)"]]}, axis=1)
display(summary)

print("\nLaporan klasifikasi model terbaik pada data uji:")
best_pred = (test_proba[best_name] >= thresholds[best_name]).astype(int)
print(classification_report(y_test, best_pred, target_names=["Asli", "Palsu"], digits=3))

## 10. Interpretasi: Kata & Fitur Paling Berpengaruh
Menggunakan koefisien Logistic Regression (Model A) agar kita tahu *mengapa* sebuah lowongan dicurigai — berguna untuk laporan & penjelasan sistem.

In [ ]:
lr = fitted["A. Logistic Regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coef = lr.named_steps["clf"].coef_.ravel()
imp = pd.Series(coef, index=[n.split("__", 1)[1] for n in names]).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
imp.tail(20).plot.barh(ax=ax[0], color="#E45756"); ax[0].set_title("20 fitur pendorong 'PALSU'")
imp.head(20).plot.barh(ax=ax[1], color="#4C78A8"); ax[1].set_title("20 fitur pendorong 'ASLI'")
plt.tight_layout(); plt.show()

## 11. Finalisasi Model untuk Deploy
Model terbaik dilatih ulang pada **seluruh data** (train + test) agar memanfaatkan data sebanyak mungkin, dengan threshold hasil cross-validation.
Set `REFIT_ON_ALL_DATA = False` bila Anda ingin men-deploy tepat model yang sudah diuji pada hold-out.

In [ ]:
REFIT_ON_ALL_DATA = True

final_model = clone(models[best_name])
if REFIT_ON_ALL_DATA:
    final_model.fit(X, y)
else:
    final_model = fitted[best_name]

bundle = {
    "model": final_model,                          # Pipeline lengkap: data mentah -> probabilitas
    "threshold": thresholds[best_name],            # ambang keputusan hasil tuning CV
    "model_name": best_name,
    "raw_columns": RAW_COLUMNS,
    "metrics_test_holdout": test_results.loc[best_name].to_dict(),
    "metrics_cv": cv_results.loc[best_name].to_dict(),
    "refit_on_all_data": REFIT_ON_ALL_DATA,
    "sklearn_version": sklearn.__version__,
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
}
print("Model final:", bundle["model_name"], "| threshold =", round(bundle["threshold"], 4))

## 12. Export Model (Paket Deploy)
Membuat folder `deploy_package/` berisi model, modul preprocessing, metadata, aplikasi demo (Gradio), dan `requirements.txt`.

In [ ]:
PKG = "deploy_package"
shutil.rmtree(PKG, ignore_errors=True); os.makedirs(PKG)

# 1) Model + threshold + metadata
joblib.dump(bundle, f"{PKG}/fake_job_detector.joblib", compress=3)

# 2) Modul preprocessing (WAJIB ada di folder yang sama saat model dimuat)
shutil.copy("fake_job_utils.py", f"{PKG}/fake_job_utils.py")

# 3) Metadata yang mudah dibaca
meta = {k: v for k, v in bundle.items() if k != "model"}
json.dump(meta, open(f"{PKG}/model_metadata.json", "w"), indent=2, default=str)

# 4) Dependensi (versi sama dengan yang dipakai training agar file .joblib kompatibel)
open(f"{PKG}/requirements.txt", "w").write(
    f"scikit-learn=={sklearn.__version__}\npandas\nnumpy\njoblib\ngradio\n")

# 5) Aplikasi demo Gradio
open(f"{PKG}/app.py", "w").write(r'''
import joblib
import pandas as pd
import gradio as gr
import fake_job_utils                       # harus di-import agar model bisa dimuat
from fake_job_utils import predict_postings

bundle = joblib.load("fake_job_detector.joblib")

def deteksi(title, company_profile, description, requirements, benefits,
            location, salary_range, employment_type, has_company_logo, has_questions):
    row = pd.DataFrame([{
        "title": title, "company_profile": company_profile, "description": description,
        "requirements": requirements, "benefits": benefits, "location": location,
        "salary_range": salary_range or None, "employment_type": employment_type or None,
        "has_company_logo": int(has_company_logo), "has_questions": int(has_questions),
    }])
    r = predict_postings(bundle, row).iloc[0]
    return f"{r['prediksi']}  |  probabilitas palsu = {r['probabilitas_palsu']:.1%}  |  risiko {r['tingkat_risiko']}"

demo = gr.Interface(
    fn=deteksi,
    inputs=[gr.Textbox(label="Judul lowongan"),
            gr.Textbox(label="Profil perusahaan", lines=3),
            gr.Textbox(label="Deskripsi pekerjaan", lines=6),
            gr.Textbox(label="Persyaratan", lines=3),
            gr.Textbox(label="Benefit", lines=2),
            gr.Textbox(label="Lokasi (mis. US, NY, New York)"),
            gr.Textbox(label="Rentang gaji (opsional)"),
            gr.Textbox(label="Tipe pekerjaan (Full-time/Part-time/...)"),
            gr.Checkbox(label="Ada logo perusahaan"),
            gr.Checkbox(label="Ada pertanyaan screening")],
    outputs=gr.Textbox(label="Hasil deteksi"),
    title="Deteksi Lowongan Kerja Palsu",
    description="Tempel isi lowongan (bahasa Inggris, sesuai dataset pelatihan) untuk dicek.")

if __name__ == "__main__":
    demo.launch()
''')

# 6) README singkat
open(f"{PKG}/README.md", "w").write(f'''# Paket Deploy - Deteksi Lowongan Kerja Palsu
Model: {bundle["model_name"]} | Threshold: {bundle["threshold"]:.4f}

## Cara pakai (Python)
    import joblib, pandas as pd
    import fake_job_utils
    from fake_job_utils import predict_postings
    bundle = joblib.load("fake_job_detector.joblib")
    df = pd.DataFrame([{{"title": "...", "description": "...", "company_profile": "..."}}])
    print(predict_postings(bundle, df))

## Jalankan demo web
    pip install -r requirements.txt
    python app.py
''')

# 7) Zip untuk diunduh
shutil.make_archive("deploy_package", "zip", PKG)
print("Isi paket:", sorted(os.listdir(PKG)))
print("Ukuran zip: %.1f MB" % (os.path.getsize("deploy_package.zip") / 1e6))

## 13. Uji Paket Deploy (simulasi di lingkungan baru)
Memuat model dari file yang sudah di-export dan memprediksi lowongan baru — persis seperti yang akan dilakukan sistem Anda nanti.

In [ ]:
loaded = joblib.load(f"{PKG}/fake_job_detector.joblib")
print("Model dimuat:", loaded["model_name"], "| threshold:", round(loaded["threshold"], 4))

contoh = pd.DataFrame([
    {   # lowongan yang tampak sah
        "title": "Senior Software Engineer",
        "company_profile": "Acme Systems is a 200-person software company building logistics tools for retailers.",
        "description": "We are looking for an engineer to design and maintain backend services in Python and Go. "
                       "You will work with a team of 8 engineers, participate in code reviews and on-call rotation.",
        "requirements": "5+ years of experience, BSc in Computer Science, experience with PostgreSQL and AWS.",
        "benefits": "Health insurance, 401k, 25 days of paid leave.",
        "location": "US, CA, San Francisco", "employment_type": "Full-time",
        "has_company_logo": 1, "has_questions": 1},
    {   # pola khas penipuan
        "title": "Work From Home - Earn $5000 Weekly!!!",
        "description": "No experience needed! Earn money fast from home. Just send your bank details and a small "
                       "registration fee to start immediately. Contact us on WhatsApp now!!!",
        "location": "US", "employment_type": "Part-time",
        "has_company_logo": 0, "has_questions": 0},
])
hasil = predict_postings(loaded, contoh)
display(pd.concat([contoh[["title"]], hasil], axis=1))

## 14. Download Paket Deploy

In [ ]:
try:
    from google.colab import files
    files.download("deploy_package.zip")
except ImportError:
    print("Berkas tersedia di:", os.path.abspath("deploy_package.zip"))

## 15. (Opsional) Jalankan Demo Web Langsung di Colab
Hapus tanda `#` pada baris di bawah untuk memunculkan antarmuka Gradio dengan link publik sementara.

In [ ]:
# !pip -q install gradio
# import sys; sys.path.insert(0, PKG)
# os.chdir(PKG)
# exec(open("app.py").read().replace('if __name__ == "__main__":', 'if True:').replace("demo.launch()", "demo.launch(share=True, debug=False)"))

## 16. Ringkasan & Langkah Berikutnya
- **Hasil**: lihat tabel perbandingan (bagian 7–9) untuk memilih model; model terbaik otomatis diekspor.
- **Deploy**: salin isi `deploy_package/` ke server/aplikasi Anda (FastAPI, Flask, Streamlit, Hugging Face Spaces, dsb.). Cukup `joblib.load` lalu `predict_postings`.
- **Keterbatasan**: dataset berbahasa Inggris & berasal dari satu periode/sumber, sehingga performa pada lowongan berbahasa Indonesia atau pola penipuan baru perlu validasi ulang / pelatihan ulang dengan data lokal.
- **Pengembangan lanjut**: tuning hyperparameter (`RandomizedSearchCV`), model IndoBERT/DistilBERT untuk teks, SMOTE/under-sampling, monitoring *drift*, dan human-in-the-loop untuk kasus berprobabilitas menengah.